In [1]:
print("Hello World")

Hello World


In [2]:
import tensorflow as tf

In [4]:
from tensorflow.keras.applications import DenseNet121, ResNet50V2
model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

model2 = ResNet50V2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 51s 1us/step


In [5]:
model.trainable = False
model2.trainable = False

In [6]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

TRAIN_DIR = r"C:\AI\Medical\Liver\data\pneumonias\train"
VAL_DIR   = r"C:\AI\Medical\Liver\data\pneumonias\val"
TEST_DIR  = r"C:\AI\Medical\Liver\data\pneumonias\test"

In [7]:
train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED
)

valid_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

print(train_raw.class_names)

Found 2409 files belonging to 2 classes.
Found 802 files belonging to 2 classes.
Found 806 files belonging to 2 classes.


['Bacterial Pneumonia', 'Viral Pneumonia']


In [ ]:
preprocess = tf.keras.applications.resnet50.preprocess_input

train = train_raw.map(
    lambda x, y: (preprocess(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

valid = valid_raw.map(
    lambda x, y: (preprocess(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

test = test_raw.map(
    lambda x, y: (preprocess(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

train = train.prefetch(tf.data.AUTOTUNE)
valid = valid.prefetch(tf.data.AUTOTUNE)
test = test.prefetch(tf.data.AUTOTUNE)

In [13]:
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from tensorflow.keras.models import Model
model2.trainable = False

x = model2.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)

output = Dense(
    1,
    activation="sigmoid"
)(x)

MODEL = Model(
    inputs=model2.input,
    outputs=output
)

In [14]:
MODEL.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_conv[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_preac… │ (None, 56, 56,    │        256 │ pool1_pool[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_preac… │ (None, 56, 56,    │          0 │ conv2_block1_pre… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,096 │ conv2_block1_pre… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_pad  │ (None, 58, 58,    │          0 │ conv2_block1_1_r… │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_2_p… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_pre… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_out    │ (None, 56, 56,    │          0 │ conv2_block1_0_c

 Total params: 23,566,849 (89.90 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 23,564,800 (89.89 MB)

In [15]:
preprocess = tf.keras.applications.resnet50.preprocess_input

train = train_raw.map(
    lambda x, y: (preprocess(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

valid = valid_raw.map(
    lambda x, y: (preprocess(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

test = test_raw.map(
    lambda x, y: (preprocess(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

train = train.prefetch(tf.data.AUTOTUNE)
valid = valid.prefetch(tf.data.AUTOTUNE)
test = test.prefetch(tf.data.AUTOTUNE)

In [16]:
MODEL.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [18]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7
    )
]

In [19]:
history = MODEL.fit(
    train,
    validation_data=valid,
    epochs=100,
    callbacks=callbacks
)

Epoch 1/100


c:\Users\smitp\.conda\envs\Medical\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


76/76 ━━━━━━━━━━━━━━━━━━━━ 49s 608ms/step - accuracy: 0.5127 - loss: 104.5258 - val_accuracy: 0.4888 - val_loss: 24.0094 - learning_rate: 1.0000e-04
Epoch 2/100
17/76 ━━━━━━━━━━━━━━━━━━━━ 25s 430ms/step - accuracy: 0.4743 - loss: 105.6697

KeyboardInterrupt: 